In [ ]:
%%writefile test.py
from pathlib import Path

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import altair as alt


# 1. Page Configuration
st.set_page_config(
    page_title="Jobs Dashboard",
    page_icon="",
    layout="wide",
    initial_sidebar_state="expanded"
)

# 2. Reading Clean Job Data (Cached for Performance)
DATA_PATH = Path(__file__).resolve().parent / "SGJob_ITSalaryClean.csv"

@st.cache_data
def load_data():
    return pd.read_csv(
        DATA_PATH,
        parse_dates=[
            "metadata_newPostingDate",
            "metadata_originalPostingDate",
            "metadata_expiryDate",
    ],
)

df_sal = load_data()

# 3. Sidebar Filters
st.sidebar.header("🕹️ Global Dashboard Filters")

# Date Filter
min_date = df_sal["metadata_newPostingDate"].min().to_pydatetime()
max_date = df_sal["metadata_newPostingDate"].max().to_pydatetime()
selected_dates = st.sidebar.date_input(
    "Select Date Range",
    value=(min_date, max_date),
    min_value=min_date,
    max_value=max_date
)

# Multi-select filters
all_type = df_sal["employmentTypes"].unique().tolist()
selected_type = st.sidebar.multiselect("Filter by Employment Types", all_type, default=all_type)

all_categories = df_sal["category"].unique().tolist()
selected_categories = st.sidebar.multiselect("Filter by Category", all_categories, default=all_categories)

all_pos = df_sal["positionLevels"].unique().tolist()
selected_pos = st.sidebar.multiselect("Filter by Position Levels", all_pos, default=all_pos)

# 4. Apply Filters to Dataset
# Handle date edge cases safely
if isinstance(selected_dates, tuple) and len(selected_dates) == 2:
    start_date, end_date = pd.to_datetime(selected_dates[0]), pd.to_datetime(selected_dates[1])
else:
    start_date, end_date = pd.to_datetime(min_date), pd.to_datetime(max_date)

filtered_df = df_sal[
    (df_sal["metadata_newPostingDate"] >= start_date) & 
    (df_sal["metadata_newPostingDate"] <= end_date) & 
    (df_sal["category"].isin(selected_categories)) & 
    (df_sal["positionLevels"].isin(selected_pos)) & 
    (df_sal["employmentTypes"].isin(selected_type))
]

# 5. Dashboard Header
st.title("Jobs Overview Dashboard")
st.markdown("Metrics for IT Jobs")
st.divider()

# 6. Top Row Layout: Metric Cards
if not filtered_df.empty:
    total_sales = filtered_df['average_salary_clean'].median()
    total_units = filtered_df['average_salary_clean'].mean()
    total_profit = filtered_df['average_salary_clean'].max()
    profit_margin = (filtered_df.groupby(pd.Grouper(key="metadata_newPostingDate", freq='M'))['numberOfVacancies'].sum()).mean()

    col1, col2, col3, col4 = st.columns(4)
    col1.metric(label="💰 Average Salary (Median)", value=f"${total_sales:,.0f}", delta="+8.3% vs Last Month")
    col2.metric(label="📦 Average Salary (Mean)", value=f"${total_units:,.0f}", delta="+4.1%")
    col3.metric(label="📈 Highest Average Salary", value=f"${total_profit:,.0f}", delta="+12.5%")
    col4.metric(label="🎯 Average Monthly Vacancies", value=f"{profit_margin:,.0f}", delta="0.8%")
else:
    st.warning("No data available for the selected filters.")

st.divider()

# 7. Middle Row Layout: Core Analytics Charts
chart_col1, chart_col2 = st.columns(2)

with chart_col1:
    st.subheader("📈 Number of Vacancies Over Time")
    if not filtered_df.empty:
        # Aggregate data by date
        trend_df = filtered_df.groupby("metadata_newPostingDate")["numberOfVacancies"].sum().reset_index()
        fig_trend = px.line(trend_df, x="metadata_newPostingDate", y="numberOfVacancies", template="plotly_white",
                            labels={"numberOfVacancies": "Vacancies", "metadata_newPostingDate": "Posting Date"})
        fig_trend.update_layout(margin=dict(l=20, r=20, t=10, b=20))
        st.plotly_chart(fig_trend, use_container_width=True)

with chart_col2:
    st.subheader("🍕 Average Salary per Position Level")
    if not filtered_df.empty:
        region_df = filtered_df.groupby("positionLevels")["average_salary_clean"].median().reset_index()
        fig_pie = px.pie(region_df, values="average_salary_clean", names="positionLevels", hole=0.4,
                         color_discrete_sequence=px.colors.sequential.RdBu)
        fig_pie.update_layout(margin=dict(l=20, r=20, t=10, b=20))
        st.plotly_chart(fig_pie, use_container_width=True)

st.divider()

# 8. Bottom Row Layout: Categorical Performance & Raw Data Table
lower_col1, lower_col2 = st.columns([3, 2])

with lower_col1:
    st.subheader("📊 Average Salary Top 10 categories")
    if not filtered_df.empty:
        cat_df = filtered_df.groupby("category")["average_salary_clean"].median().sort_values(ascending=False).reset_index().head(10)
        fig_bar = px.bar(cat_df, x="average_salary_clean", y="category", orientation="h", 
                         text_auto=".2s", color="average_salary_clean",
                         color_continuous_scale=px.colors.sequential.Viridis,labels={"average_salary_clean": "Average Salary"})
        fig_bar.update_layout(margin=dict(l=20, r=20, t=10, b=20))
        st.plotly_chart(fig_bar, use_container_width=True)

with lower_col2:
    st.subheader("🔍 Category Position Salary Heatmap")
    if not filtered_df.empty:
        def make_heatmap(input_df, input_y, input_x, input_color, input_color_theme):
            heatmap = alt.Chart(input_df).mark_rect().encode(
                    y=alt.Y(f'{input_y}:O', axis=alt.Axis(title="", titleFontSize=18, titlePadding=15, titleFontWeight=900, labelAngle=0)),
                    x=alt.X(f'{input_x}:O', axis=alt.Axis(title="", titleFontSize=18, titlePadding=15, titleFontWeight=900)),
                    color=alt.Color(f'max({input_color}):Q',
                             legend=None,
                             scale=alt.Scale(scheme=input_color_theme)),
                    stroke=alt.value('black'),
                    strokeWidth=alt.value(0.25),
                ).properties(width=900
                ).configure_axis(
                labelFontSize=12,
                titleFontSize=12
                ) 
            # height=300
            return heatmap

        colors = ['blues', 'cividis', 'greens', 'inferno', 'magma', 'plasma', 'reds', 'rainbow', 'turbo', 'viridis']
        heatmap = make_heatmap(filtered_df, 'positionLevels', 'employmentTypes', 'average_salary_clean', 'blues')
        st.altair_chart(heatmap, use_container_width=True, height="stretch")



Overwriting test.py
